- 범주형 변수를 전처리 없이 대부분의 머신러닝 모델에 넣으면 **에러 발생**
- 범주형 데이터를 준비하는 세 가지 접근 방식 비교
---

### 학습 데이터 확인 및 범주형 변수 확인

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 로드
melbourne_data = pd.read_csv('data/melb_data.csv')
y = melbourne_data.Price


melbourne_features = ['Type','Method','Regionname','Rooms','Distance','Postcode','Bedroom2', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude','Propertycount']
X = melbourne_data[melbourne_features]


# 훈련 데이터와 검증 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)


# 범주형 변수 확인
s = (X_train.dtypes == 'object')
object_cols = list(s[s].index)


print("Categorical variables:")
print(object_cols)


Categorical variables:
['Type', 'Method', 'Regionname']


### [방법1] : 범주형 변수 제거하기
- 데이터 세트에서 단순히 제거하는 방법
- 해당 컬럼이 유용한 정보를 포함하고 있지 않을 때

In [17]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

# 모델 평가 함수 정의
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 모델 성능 평가 함수
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop categorical variables):
196095.82210550059



### [방법2] : 순서 인코딩 (Ordinal Encoding)
- 각 고유 값에 다른 정수 할당하는 방법으로 열에 유용한 정보가 포함되어 있지 않은 경우
- **결정 트리, 랜덤 포레스트** 같은 트리 기반 모델에서 잘 작동

In [18]:
from sklearn.preprocessing import OrdinalEncoder

label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])


print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE from Approach 2 (Ordinal Encoding):
179956.4228995778


### [방법3] : 원-핫 인코딩 (One-Hot Encoding)
- 각 범주에 대한 **별도의 열을 생성** 하여, 각 범주의 존재 여부(1 또는 0) 으로 표현
- 범주 간 순서가 없을 때 유용하여 너무 많은 고유 값을 가진 변수에는 원-핫 인코딩이 잘 작동하지 않을 수 있다. 

In [22]:
from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE from Approach 3 (One-Hot Encoding):
178881.11787045436


### 결론 
- **범주형 변수를 제거한 방법(방법1)** 이 가장 높은 MAE를 기록하며 가장 성능이 낮았다.
- 일반적으로 원-핫 인코딩 > 순서 인코딩 > 범주형 제거 순으로 효과적이다.

In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 로드
X = pd.read_csv('data/train.csv', index_col='Id') 
X_test = pd.read_csv('data/test.csv', index_col='Id')

# 누락된 타겟 값이 있는 행 제거, 타겟 값 분리
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

# 누락된 값이 있는 열 제거
cols_with_missing = [col for col in X.columns if X[col].isnull().any()] 
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

# 훈련 데이터와 검증 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)


# 모델 평가 함수 정의
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 모델 성능 평가 함수
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)



[방법1] : 범주형 데이터가 포함된 열 삭제

In [39]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))


MAE from Approach 1 (Drop categorical variables):
17837.82570776256


[방법2] : 순서형(Ordinal) 인코딩

In [40]:
print("훈련 데이터에서 'Condition2'의 고유 값:", X_train['Condition2'].unique())
print("검증 데이터에서 'Condition2'의 고유 값:", X_valid['Condition2'].unique())

훈련 데이터에서 'Condition2'의 고유 값: ['Norm' 'PosA' 'Feedr' 'PosN' 'Artery' 'RRAe']
검증 데이터에서 'Condition2'의 고유 값: ['Norm' 'RRAn' 'RRNn' 'Artery' 'Feedr' 'PosN']


검증 데이터에는 훈련 데이터에 없는 'RRAn'과 'RRNn'이 포함되어 있어, 일반적인 순서형 인코더는 이를 처리하지 못하고 오류를 발생시킵니다.

In [41]:
from sklearn.preprocessing import OrdinalEncoder

# 문제 있는 열 찾기
object_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]
good_label_cols = [col for col in object_cols if set(X_valid[col]).issubset(set(X_train[col]))]
bad_label_cols = list(set(object_cols) - set(good_label_cols))

print('Categorical columns that will be ordinal encoded:', good_label_cols)
print('Categorical columns that will be dropped from the dataset:', bad_label_cols,'/n')


# 순서형 인코딩 적용
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

ordinal_encoder = OrdinalEncoder()
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(X_train[good_label_cols])
label_X_valid[good_label_cols] = ordinal_encoder.transform(X_valid[good_label_cols])


print("\nMAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

Categorical columns that will be ordinal encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'PavedDrive', 'SaleType', 'SaleCondition']
Categorical columns that will be dropped from the dataset: ['Functional', 'Condition2', 'RoofMatl'] /n

MAE from Approach 2 (Ordinal Encoding):
17098.01649543379


[방법3] : 범주형 변수의 고유 값 개수(Cardinality) 조사

고유 값이 많은 열(고차원 범주형 변수)은 원-핫 인코딩 시 데이터 크기를 과도하게 증가시킬 수 있습니다.

In [42]:
# 각 범주형 열의 고유 값 개수 확인
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# 결과 정렬 및 출력
sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('Utilities', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('LotShape', 4),
 ('LandContour', 4),
 ('ExterQual', 4),
 ('KitchenQual', 4),
 ('MSZoning', 5),
 ('LotConfig', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('HeatingQC', 5),
 ('Condition2', 6),
 ('RoofStyle', 6),
 ('Foundation', 6),
 ('Heating', 6),
 ('Functional', 6),
 ('SaleCondition', 6),
 ('RoofMatl', 7),
 ('HouseStyle', 8),
 ('Condition1', 9),
 ('SaleType', 9),
 ('Exterior1st', 15),
 ('Exterior2nd', 16),
 ('Neighborhood', 25)]

[방법4] : 원-핫 인코딩

고유 값이 10개 미만인 열(low_cardinality_cols)을 원-핫 인코딩하고, 그렇지 않은 열(high_cardinality_cols)은 삭제합니다.

In [44]:
from sklearn.preprocessing import OneHotEncoder

# 채워야 할 부분: 훈련 데이터에서 cardinality가 10보다 큰 범주형 변수의 개수는 몇 개인가?
high_cardinality_numcols = 3

# 채워야 할 부분: 'Neighborhood' 변수의 원-핫 인코딩을 위해 몇 개의 열이 필요한가?
num_cols_neighborhood = 25

OH_entries_added = 1e4*100 - 1e4

label_entries_added = 0

# 원-핫 인코딩할 열 선택 (Cardinality < 10)
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < 10]

# 삭제할 범주형 열 선택 (Cardinality >= 10)
high_cardinality_cols = list(set(object_cols) - set(low_cardinality_cols))

print('원-핫 인코딩할 범주형 열:', low_cardinality_cols)
print('삭제할 범주형 열:', high_cardinality_cols)

# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols]))

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

원-핫 인코딩할 범주형 열: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']
삭제할 범주형 열: ['Exterior2nd', 'Neighborhood', 'Exterior1st']


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'